# Script for Table Preperation of NGS data

## 0. Pre-adjusted settings
### 0.1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import gc
from Bio.Seq import Seq
from itertools import product
import inspect
import matplotlib as plt
import matplotlib.pyplot as plt


### 0.2. Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for NGS_tissue
NGS_tissue_dir = NGS_dir/'tissues'
os.makedirs(NGS_tissue_dir, exist_ok=True)

# Path for NGS_input
NGS_input_dir = NGS_dir/'input_lib'
os.makedirs(NGS_input_dir, exist_ok=True)

# Path for tech_rep
NGS_tech_rep_dir = NGS_dir/'tech_rep'
os.makedirs(NGS_tech_rep_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for plots
plots_dir = general_dir/'figures'
os.makedirs(plots_dir, exist_ok=True)


### 0.3 Definition of functions for processing the tables

#### 0.3.1. extract sample folder names

In [ ]:
def ext_sample_folders(folder_dir):
    folders = sorted([
        f.name 
        for f in folder_dir.iterdir() 
        if not f.name.startswith(".")
    ])
    return folders

#### 0.3.2. Create lists based on sample folders

In [ ]:
def extract_lists (folders):
    tissues = []
    ext_types = []

    for folder in folders:
        parts = folder.split("_")

        ext, tissue = parts

        if ext not in ext_types:
            ext_types.append(ext)

        if tissue not in tissues:
            tissues.append(tissue)

    return tissues, ext_types

#### 0.3.3. Load csv files

In [ ]:
def load_csv_files(path, folders, mouse_ids):
    """
    Load CSV files into nested dict:
    NGS_files[tissue][ext][mouse_id] = dataframe
    """

    path = Path(path)
    NGS_files = {}

    for folder in folders:
        folder_path = path / folder

        parts = folder.split("_")

        ext = parts[0]     # cDNA or gDNA
        tissue = parts[1]  # liver or heart

        NGS_files.setdefault(tissue, {}).setdefault(ext, {})

        # ext-specific prefix
        if ext == "cDNA":
            prefix = "c"
        elif ext == "gDNA":
            prefix = "g"

        # alle csv files im Ordner
        csv_files = [f for f in folder_path.iterdir() if f.is_file() and f.suffix.lower() == ".csv"]

        for mouse_id in mouse_ids:
            matching_files = []

            for file in csv_files:
                name_lower = file.name.lower()

                if name_lower.startswith(prefix) and mouse_id.lower() in name_lower:
                    matching_files.append(file)
                    
            df = pd.read_csv(matching_files[0])
            NGS_files[tissue][ext][mouse_id] = df

    return NGS_files
            

#### 0.3.4. Load input librarys for tissue and extraction_type

In [ ]:
# Load input library
def load_input_library ():
    df = pd.read_csv(NGS_input_dir / 'input_lib_outputg35_input_lib_S35_R1_001_PV.csv')
    input_library = {}
    for tissue in TISSUES:
        input_library[tissue] = {}
        for ext in EXT:
            input_library[tissue][ext] = df.copy()
    return input_library

#### 0.3.5. translate nt --> AA sequence

In [ ]:
def translate_to_AA(files):
    AA_files = {}

    for tissue in TISSUES:
        AA_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                df_aa = (
                    df.groupby("AA_sequence", as_index=False)['Count']
                    .sum()
                    .sort_values('Count', ascending=False)
                    .reset_index(drop=True)
                )

                AA_files[tissue][ext] = df_aa

            # Check if file is sample with Mouse_ID
            elif isinstance(item, dict):
                AA_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()
                    df["AA_sequence"] = df['Peptide'].apply(lambda x: str(Seq(x).translate()))

                    df_aa = (
                        df.groupby("AA_sequence", as_index=False)['Count']
                        .sum()
                        .sort_values('Count', ascending=False)
                        .reset_index(drop=True)
                    )

                    AA_files[tissue][ext][sample_key] = df_aa

    return AA_files


#### 0.3.6. Remove AA_seq with stop codon

In [ ]:
def remove_stop_codon (files):
    AA_no_stop_files = {}

    for tissue in TISSUES:
        AA_no_stop_files[tissue]= {}

        for ext in EXT:
            item = files[tissue][ext]

            # check if file input_library 
            if isinstance(item, pd.DataFrame):
                df = item.copy()
                df = df[~df["AA_sequence"].str.contains(r"\*")]
                
                AA_no_stop_files[tissue][ext] = df
            # check if dict (Samples)
            elif isinstance(item, dict):
                AA_no_stop_files[tissue][ext] = {}
                
                for sample_key, df in item.items():
                    df = df.copy()
                    df = df[~df["AA_sequence"].str.contains(r"\*")]
                    
                    AA_no_stop_files[tissue][ext][sample_key] = df
                
    return AA_no_stop_files

#### 0.3.7. Merge AA_seq together RENAME

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
def create_tissue_AA_string(files):
    AA_tissue_string = {}

    for tissue in TISSUES:
        all_aa = []

        for ext in EXT:
            for mouse_ID in MOUSE_ID:
                df = files[tissue][ext][mouse_ID].copy()
                all_aa.extend(df["AA_sequence"].tolist())

        # unique AA sequences pro Tissue
        AA_tissue_string[tissue] = pd.Series(all_aa).drop_duplicates().tolist()

    return AA_tissue_string

#### 0.3.7. Create Pseudo library and frame shift Merge will function before

In [ ]:
def expand_input_library_with_sample_AA(long_table, input_lib):
    expanded_input = {}

    for tissue in TISSUES:
        expanded_input[tissue] = {}

        for ext in EXT:
            aa_sample = long_table[tissue].copy()

            #input library
            df_input = input_lib[tissue][ext].copy()

            # Outer merge mit allen AA_seq aus Samples
            df_merged = pd.DataFrame({"AA_sequence": aa_sample}).merge(
                df_input,
                on="AA_sequence",
                how="outer"
            )
            
            # wenn Count fehlt -> neu durch Samples hinzugefügt
            df_merged["Pseudo"] = 0
            mask = df_merged["Count"].isna()
            df_merged.loc[mask, "Pseudo"] = 1
            
            df_merged["Count"] = df_merged["Count"].fillna(0)
            
            df_merged = (
                df_merged
                .sort_values("Count", ascending=False)
                .reset_index(drop=True)
            )
            df_merged['Count'] = df_merged['Count']+1
            expanded_input[tissue][ext] = df_merged
            

    return expanded_input

#### 0.3.8. Create Pseudo samples with frameshift

In [ ]:
def merge_samples_with_input(files, input_lib):
    merged_files = {}

    for tissue in TISSUES:
        merged_files[tissue] = {}

        for ext in EXT:
            merged_files[tissue][ext] = {}

            # input library for tissue and ext
            df_input = input_lib[tissue][ext].copy()
            df_input = df_input[['AA_sequence', 'Count', 'Pseudo', 'RPM']]

            # rename input columns
            rename_dict = {
                "Count": "Count_input",
                "Pseudo": "Pseudo_input",
                "RPM": "RPM_input"
            }
            df_input = df_input.rename(columns=rename_dict)

            for mouse_ID in MOUSE_ID:
                df_sample = files[tissue][ext][mouse_ID].copy()

                # merge sample + input
                df_merged = df_sample.merge(
                    df_input,
                    on="AA_sequence",
                    how="outer"
                )

                # NaN = 0
                df_merged["Pseudo"] = 0
                mask = df_merged["Count"].isna()
                df_merged.loc[mask, "Pseudo"] = 1
                df_merged = df_merged.fillna(0)

                global_factor = df_merged["Count"].sum() / df_merged["Count_input"].sum()
                df_merged['Count'] = df_merged['Count'] + global_factor

                merged_files[tissue][ext][mouse_ID] = df_merged

    return merged_files

#### 0.3.9. Calculate 

##### 0.3.9.1 Calculate RPM and Log2_enrichment for gDNA and cDNA

In [ ]:
def calculate_tables (files):
    AA_calc_files = {}

    for tissue in TISSUES:
        AA_calc_files[tissue] = {}

        for ext in EXT:
            item = files[tissue][ext].copy()

            # if input library
            if isinstance(item, pd.DataFrame):
                df = item.copy()

                total_count = df["Count"].sum()
                
                df["RPM"] = (df["Count"] / total_count)*1e+06
                df["Cum_RPM"] = df["RPM"].cumsum()

                df = df.reset_index(drop=True)

                AA_calc_files[tissue][ext] = df
                del files[tissue][ext]

            # if sample with mouse_ID
            elif isinstance(item, dict):
                AA_calc_files[tissue][ext] = {}

                for sample_key, df in item.items():
                    df = df.copy()

                    total_count = df["Count"].sum()

                    df["RPM"] = (df["Count"] / total_count)*1e+06
                    df["Cum_RPM"] = df["RPM"].cumsum()
                    df['Log2_enrichment'] = np.log2(df['RPM']/df['RPM_input'])

                    AA_calc_files[tissue][ext][sample_key] = df[['AA_sequence', 'Count', 'RPM', 'RPM_input', 'Cum_RPM', 'Log2_enrichment', 'Pseudo_input', 'Pseudo']].sort_values("Log2_enrichment", ascending=False).reset_index(drop=True)
                    del files[tissue][ext][sample_key]

    gc.collect()
    
    return AA_calc_files

##### 0.3.9.2 Calculate Log2_enrichment from gDNA to cDNA,

In [ ]:
def calculate_log2_enrichment_from_gDNA_to_cDNA(files):
    
    dict_new = {}
    for tissue, mouse_id in product(TISSUES, MOUSE_ID):
        
        df_cDNA = files[tissue]['cDNA'][mouse_id].copy()
        
        df_gDNA = files[tissue]['gDNA'][mouse_id][['AA_sequence', 'RPM', 'Pseudo']].rename(
            columns={
                'RPM': 'RPM_gDNA',
                'Pseudo': 'Pseudo_gDNA'}
        ).copy()

        df_merge = df_cDNA.merge(
            df_gDNA,
            on='AA_sequence',
            how='outer'
        )

        df_merge['Log2_enrichment_gDNA_to_cDNA'] = np.log2(
            df_merge['RPM'] / df_merge['RPM_gDNA']
        )
        df_merge = df_merge.drop(columns =['Pseudo_gDNA'])
        df_gDNA_new = files[tissue]['gDNA'][mouse_id]
        
        dict_new.setdefault(tissue, {}).setdefault('cDNA', {})[mouse_id] = df_merge
        
        dict_new.setdefault(tissue, {}).setdefault('gDNA', {})[mouse_id] = df_gDNA_new

    return dict_new


#### 0.3.10. Create long table with all samples

In [ ]:
def create_long_table (tables):

    value_columns = ['Count', 'RPM', 'RPM_input', 'Cum_RPM', 'Log2_enrichment', 'RPM_gDNA', 'Log2_enrichment_gDNA_to_cDNA', 'Pseudo_input', 'Pseudo']
                    
    # Create new columns for sample identification 
    df_long = []
    for tissue, ext, mouse_ID in product(TISSUES, EXT, MOUSE_ID):
        df = tables[tissue][ext][mouse_ID].copy()

        df["Sample"] = f"{ext}_{tissue}_{mouse_ID}"
        df["Sex"] = get_sex(mouse_ID)
        df["Mouse_ID"] = mouse_ID
        df["Tissue"] = tissue
        df["Extraction_type"] = ext
        df_long.append(df)
    df_long_table_metadata = pd.concat(df_long, ignore_index=True)

    #Reorganization of columns, first AA_sequence, 2nd Sample type, 3rd values
    df_long_table_metadata = df_long_table_metadata[
        [c for c in df_long_table_metadata.columns if c not in value_columns] + value_columns
    ].copy()
   
    df_long_table = df_long_table_metadata[['AA_sequence', 'Sample']+ value_columns].copy()
    return df_long_table_metadata, df_long_table

#### 0.3.11. Create pooled table

In [ ]:
def create_pooled_table(long_table_metadata):
    cols = ["AA_sequence", "Tissue", "Extraction_type", "Count", "RPM", "RPM_input", "RPM_gDNA", "Log2_enrichment", "Pseudo_input", "Pseudo"]

    df_long = long_table_metadata[cols].copy()



    df_pooled = (
        df_long
        .groupby(["AA_sequence", "Tissue", "Extraction_type"], as_index=False, sort = False, observed = True)
        .agg({
            'Count': 'mean',
            "RPM": "mean",
            "RPM_input": "first",
            "RPM_gDNA": "mean",
            "Log2_enrichment": "mean",
            "Pseudo_input": "first",
            "Pseudo": "sum"
        })
    )
    # keep old log2_enrichment calculation
    df_pooled = df_pooled.rename(columns={"Log2_enrichment": "Log2_enrichment_old"})

    # number of biological samples in which variant is present
    df_pooled["n_samples_present"] = 6 - df_pooled["Pseudo"]

    df_pooled['Log2_enrichment'] = np.log2(df_pooled['RPM']/df_pooled['RPM_input'])
    
    # initialize new column with NaN
    df_pooled["Log2_enrichment_gDNA_to_cDNA"] = np.nan

    mask = (
        (df_pooled["Extraction_type"] == "cDNA") &
        (df_pooled["RPM_gDNA"] > 0)
    )
    df_pooled.loc[mask, "Log2_enrichment_gDNA_to_cDNA"] = np.log2(
        df_pooled.loc[mask, "RPM"] / df_pooled.loc[mask, "RPM_gDNA"]
    )
    
    df_pooled = df_pooled[['AA_sequence', 'Tissue', 'Extraction_type', 'Count', 'RPM', 'Log2_enrichment', 'Log2_enrichment_old', 'Log2_enrichment_gDNA_to_cDNA', 'n_samples_present', 'RPM_input', 'Pseudo_input'
                          ]].sort_values("Log2_enrichment", ascending=False).reset_index(drop=True)
    return df_pooled

In [ ]:
def create_pooled_table_sex(long_table_metadata):
    cols = ["AA_sequence", "Sex", "Tissue", "Extraction_type", "Count", "RPM", "RPM_input", "RPM_gDNA", "Log2_enrichment", "Pseudo_input", "Pseudo"]
    df_long = long_table_metadata[cols].copy()


    df_pooled_sex = (df_long
        .groupby(["AA_sequence", "Sex", "Tissue", "Extraction_type"], as_index=False, sort = False, observed = True)
        .agg({
            'Count': 'mean',
            "RPM": "mean",
            "RPM_input": "first",
            "RPM_gDNA": "mean",
            "Log2_enrichment": "mean",
            "Pseudo_input": "first",
            "Pseudo": "sum"
        })
    )

    # keep old log2_enrichment calculation
    df_pooled_sex = df_pooled_sex.rename(columns={"Log2_enrichment": "Log2_enrichment_old"})

    # number of biological samples present within each sex group

    df_pooled_sex["n_samples_present"] = 3 - df_pooled_sex["Pseudo"]

    # new pooled enrichment
    df_pooled_sex['Log2_enrichment'] = np.log2(df_pooled_sex['RPM']/df_pooled_sex['RPM_input'])

    # initialize new cDNA vs gDNA enrichment column
    df_pooled_sex["Log2_enrichment_gDNA_to_cDNA"] = np.nan

    mask = (
        (df_pooled_sex["Extraction_type"] == "cDNA") &
        (df_pooled_sex["RPM_gDNA"] > 0)
    )

    df_pooled_sex.loc[mask, "Log2_enrichment_gDNA_to_cDNA"] = np.log2(
        df_pooled_sex.loc[mask, "RPM"] / df_pooled_sex.loc[mask, "RPM_gDNA"]
    )

    df_pooled_sex = df_pooled_sex[['AA_sequence', 'Sex', 'Tissue', 'Extraction_type', 'Count', 'RPM', 'Log2_enrichment', 'Log2_enrichment_old', 'Log2_enrichment_gDNA_to_cDNA', 'n_samples_present', 'RPM_input', 'Pseudo_input']
                          ].sort_values(["Tissue", "Extraction_type", "Sex", "Log2_enrichment"],ascending=[True, True, True, False]).reset_index(drop=True)

    return df_pooled_sex

#### 0.3.12. Create pivot table

In [ ]:
def pivot_table_log (table):
    
    df_wide = (
        table.pivot_table(
            index=["AA_sequence", "Tissue"],
            columns="Extraction_type",
            values=[
                "Log2_enrichment",
                'RPM',
                'Log2_enrichment_gDNA_to_cDNA'
            ],
            aggfunc="first"
        )
    )
    
    df_wide.columns = [
    f"{a}_{b}" if b != "" else a
    for a, b in df_wide.columns
    ]
    df_wide = df_wide.reset_index()
    
    return df_wide

### 0.4. Definition of helper functions
#### 0.4.1. Save tables

In [ ]:
# Save tables
def save_tables (path, table, name):
    folder_path = path
    os.makedirs(folder_path, exist_ok=True)
    table.to_csv(folder_path / f"{name}.csv", index=False)
    

#### 0.4.2. Save plots

In [ ]:
# save plots
def save_plot(fig, path, name):
    os.makedirs(path, exist_ok=True)
    fig.savefig(os.path.join(path, f"{name}.png"),
                dpi=300,
                bbox_inches="tight")

#### 0.4.3. Check sex of mouse_ID

In [ ]:
def get_sex(mouse_id):
    mouse_id = mouse_id.lower()
    
    if mouse_id.startswith("f"):
        return "female"
    elif mouse_id.startswith("m"):
        return "male"
    else:
        return "unknown"

#### 0.4.4. Check dict NOT USED

In [ ]:
#def for checking dict

def check(dictionary, path=None):
    if path is None:
        path = []

    if isinstance(dictionary, dict):
        for key, value in dictionary.items():
            check(value, path + [key])
    else:
        if hasattr(dictionary, "shape"):
            msg = f"{' - '.join(map(str, path))}: {dictionary.shape} | {len(dictionary.columns)} columns"

            for col in dictionary.columns:
                if col.startswith("Count"):
                    msg += f" | {col} sum = {dictionary[col].sum():,.0f}" 
                    if col == 'Count_tissue':
                        count = len(dictionary[dictionary['Count_tissue'] != 0])
                    if col == 'Count_input_lib':
                        count = len(dictionary[dictionary['Count_input_lib'] != 0])
            msg += f'| variants before = {count}'
            print(msg)
        else:
            print(f"{' - '.join(map(str, path))}: {type(dictionary)}")

#### 0.4.5. find biggest dictionaries

In [ ]:
def get_size(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.memory_usage(deep=True).sum()
    elif isinstance(obj, dict):
        return sum(get_size(v) for v in obj.values())
    elif isinstance(obj, list):
        return sum(get_size(i) for i in obj)
    elif isinstance(obj, tuple):
        return sum(get_size(i) for i in obj)
    else:
        return 0

def top_dicts(n=5):
    dict_sizes = []

    for name, obj in globals().items():
        if isinstance(obj, dict):
            size = get_size(obj)
            dict_sizes.append((name, size))

    dict_sizes.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTop {n} größte Dictionaries:\n")
    for name, size in dict_sizes[:n]:
        print(f"{name:30s} {size / 1024**3:8.2f} GB")

#### 0.4.6. Function to compare log2_enrichment to log2_enrichment_old

In [ ]:
def corr_log2_enrichment (table, tissue, extraction):

    df = table[
        ["Log2_enrichment", "Log2_enrichment_old", "Tissue", "Extraction_type"]
    ].dropna().copy()
    df = df[
        (df['Tissue'] == tissue) &
        (df['Extraction_type'] == extraction)
    ].copy()

    r_pearson, _ = pearsonr(df["Log2_enrichment_old"], df["Log2_enrichment"])
    r_spearman, _ = spearmanr(df["Log2_enrichment_old"], df["Log2_enrichment"])
    
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    
    sns.scatterplot(
        data=df,
        x="Log2_enrichment_old",
        y="Log2_enrichment",
        alpha=0.25,
        s=8,
        ax=ax
    )
    
    lims = [
        min(df["Log2_enrichment_old"].min(), df["Log2_enrichment"].min()),
        max(df["Log2_enrichment_old"].max(), df["Log2_enrichment"].max())
    ]
    ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.5)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    
    ax.text(
        0.05, 0.95,
        f"Pearson r = {r_pearson:.3f}\nSpearman ρ = {r_spearman:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="0.7")
    )
    
    ax.set_xlabel("Old pooled score = mean(Log2_enrichment)")
    ax.set_ylabel("New pooled score = log2(mean RPM / mean reference RPM)")
    ax.set_title("Old vs new pooled enrichment")
    
    sns.despine()
    plt.tight_layout()
    plt.show()

### 1. Script

### 1.1. Create lists

In [ ]:
FOLDER_NAMES = ext_sample_folders (NGS_tissue_dir)
TISSUES, EXT = extract_lists(FOLDER_NAMES)
MOUSE_ID = ['f1', 'f2', 'f3', 'm1', 'm2', 'm3']

In [ ]:
FOLDER_NAMES, TISSUES, EXT

### 1.2. Load, translate and remove AA_seq with stop_codon from samples

In [ ]:
%%time
# load csv files
NGS_files = load_csv_files(NGS_tissue_dir, FOLDER_NAMES, MOUSE_ID)

# translate nt files to AA files
dict_AA_samples = translate_to_AA (NGS_files)

# remove AA_seq with stop codons
dict_AA_samples_no_stop = remove_stop_codon (dict_AA_samples)

### 1.3. Load, translate and remove AA_seq with stop_codon from input library

In [ ]:
%%time
# load librarys
dict_input_lib = load_input_library()

# translate nt files to AA files
dict_AA_input = translate_to_AA (dict_input_lib)

# remove AA_seq with stop codons
dict_AA_input_no_stop = remove_stop_codon (dict_AA_input)

#### 1.3.1. Save raw data tables with AA_seq

In [ ]:
save = False

if save:
    # Save all raw sample tables
    for tissue, ext, mouse_ID in product (TISSUES, EXT, MOUSE_ID):
        save_tables(tables_dir/tissue, dict_AA_samples_no_stop[tissue][ext][mouse_ID], f'df_{ext}_{tissue}_{mouse_ID}_raw' )
    
    # Save raw library table
    save_tables(tables_dir, dict_AA_input_no_stop['liver']['gDNA'], 'df_input_lib_raw')

### 1.4. Create tissue and gDNA/cDNA specific input library with frameshift +1

In [ ]:
# merge all mouse ID together and create pseudo AA_seq and frameshift +1 for sample and input_library
AA_seq_per_tissue = create_tissue_AA_string (dict_AA_samples_no_stop)
dict_sample_input = expand_input_library_with_sample_AA (AA_seq_per_tissue, dict_AA_input_no_stop)

# calculate prop and cum_RPM for input
AA_input_calc = calculate_tables (dict_sample_input)

In [ ]:
display (AA_input_calc['liver']['cDNA'])

### 1.5. calculate RPM and log2_enrichment for samples with frameshift + (sum(count_sample)/sum(count_input))

In [ ]:
%%time
# Merge samples with calc_input
dict_sample_merged = merge_samples_with_input (dict_AA_samples_no_stop, AA_input_calc)

# calc RPM, cumulative prop and Log2_enrichment for samples
dict_samples = calculate_tables (dict_sample_merged)

# calculate log2_enrichment from gDNA to cDNA
dict_samples_new = calculate_log2_enrichment_from_gDNA_to_cDNA (dict_samples)
del dict_samples
gc.collect()

In [ ]:
dict_samples_new['liver']['cDNA']['f2'].sort_values('Log2_enrichment', ascending = False)

### 1.6. Create long table with metadata for ploting 

In [ ]:
%%time
# create long table with metadata for ploting and one condense to save
df_long_table_metadata, df_long_table = create_long_table(dict_samples_new)

In [ ]:
df_long_table_metadata

### 1.7. Create pooled table

In [ ]:
%%time
# Create pooled table
df_pooled_tissues = create_pooled_table(df_long_table_metadata)

In [ ]:
df_pooled_tissues

### 1.8. Create sex pool table 

In [ ]:
df_pooled_sex = create_pooled_table_sex(df_long_table_metadata)

In [ ]:
df_pooled_sex

### 1.9. Create pivot table

In [ ]:
df_pivot_table = pivot_table_log(df_pooled_tissues)

In [ ]:
df_pivot_table

### 1.8. Save all tables

In [ ]:
%%time
# Save all sample tables
for tissue, ext, mouse_ID in product (TISSUES, EXT, MOUSE_ID):
    save_tables(tables_dir/tissue, dict_samples_new[tissue][ext][mouse_ID], f'df_{ext}_{tissue}_{mouse_ID}_processed' )


In [ ]:
# Save all input librarys tables
for tissue in TISSUES:
    save_tables(tables_dir/tissue, AA_input_calc[tissue]['cDNA'], f'df_{tissue}_input_library' )

In [ ]:
# Save long table metadata
save_tables(tables_dir/"summary", df_long_table_metadata, 'df_long_all_samples_metadata')

In [ ]:
# Save long table
save_tables(tables_dir/"summary", df_long_table, 'df_long_all_samples')

In [ ]:
# Save pooled table
save_tables(tables_dir/"summary", df_pooled_tissues, 'df_sample_pooled')

In [ ]:
# Save pooled sex table
save_tables(tables_dir/"summary", df_pooled_sex, 'df_sample_pooled_sex')

In [ ]:
# Save pooled_pivot_log_enrichment table
save_tables(tables_dir/"summary", df_pivot_table, 'df_pooled_pivot')

# DONE